In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
import sys
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import BertTokenizer

sys.path.insert(0, r"d:\study\ml-foundations\bert")
from paralla_decoder import BertEncoder, ParallelDecoder

/usr/local/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


using: cuda


In [3]:
class SequencePooler(nn.Module):
    """Mean-pools BERT hidden states [B, seq_len, hidden_dim] → [B, latent_dim]."""
    def __init__(self, hidden_dim=768, latent_dim=256):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, latent_dim)

    def forward(self, hidden_states, attention_mask=None):
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        else:
            pooled = hidden_states.mean(1)
        return self.proj(pooled)   # [B, latent_dim]

In [4]:
encoder = BertEncoder().to(device)
decoder = ParallelDecoder(latent_dim=256).to(device)
print("encoder and decoder ready")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13148.08it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15361.77it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  

encoder and decoder ready


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MetricNet(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, latent_dim * latent_dim)
        )
    
    def forward(self, z):
        B = z.shape[0]
        L_flat = self.net(z)                                    # [B, d*d]
        L = L_flat.view(B, self.latent_dim, self.latent_dim)   # [B, d, d]
        
        # enforce lower triangular
        L = torch.tril(L)
        
        # enforce positive diagonal via softplus
        diag_idx = torch.arange(self.latent_dim)
        L = L.clone()
        L[:, diag_idx, diag_idx] = F.softplus(L[:, diag_idx, diag_idx]) + 1e-4
        
        # SPD matrix
        G = L @ L.transpose(-2, -1)                            # [B, d, d]
        return G, L


# ── test ──────────────────────────────────────────────────────────────
metric_net = MetricNet(latent_dim=256).to(device)

z_test = torch.randn(2, 256).to(device)
G, L = metric_net(z_test)

eigenvalues = torch.linalg.eigvalsh(G)

print(f"G shape:        {G.shape}")           # [2, 256, 256]
print(f"L shape:        {L.shape}")           # [2, 256, 256]
print(f"min eigenvalue: {eigenvalues.min().item():.6f}")  # must be > 0
print(f"max eigenvalue: {eigenvalues.max().item():.6f}")
print(f"SPD check:      {'PASSED' if eigenvalues.min() > 0 else 'FAILED'}")

G shape:        torch.Size([2, 256, 256])
L shape:        torch.Size([2, 256, 256])
min eigenvalue: 0.012813
max eigenvalue: 3.287915
SPD check:      PASSED


In [6]:
class FlowNet(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, 512),  # +1 for time t
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, latent_dim)
        )
    
    def forward(self, z_t, t):
        # z_t: [B, latent_dim]
        # t:   [B]
        t_expand = t.unsqueeze(-1)                  # [B, 1]
        inp      = torch.cat([z_t, t_expand], dim=-1)  # [B, latent_dim+1]
        return self.net(inp)                        # [B, latent_dim]


# ── test ──────────────────────────────────────────────────────────────
flow_net = FlowNet(latent_dim=256).to(device)

z_t_test = torch.randn(2, 256).to(device)
t_test   = torch.rand(2).to(device)

v_pred = flow_net(z_t_test, t_test)

print(f"z_t shape: {z_t_test.shape}")  # [2, 256]
print(f"t shape:   {t_test.shape}")    # [2]
print(f"v_pred shape: {v_pred.shape}") # [2, 256]
print("FlowNet check: PASSED" if v_pred.shape == z_t_test.shape else "FAILED")

z_t shape: torch.Size([2, 256])
t shape:   torch.Size([2])
v_pred shape: torch.Size([2, 256])
FlowNet check: PASSED


In [7]:
checkpoint = torch.load("stage1_best.pt", map_location=device)
decoder.load_state_dict(checkpoint["decoder"])
for param in decoder.parameters():
    param.requires_grad = False
print("stage1 decoder loaded and frozen")

stage1 decoder loaded and frozen


In [8]:
from paralla_decoder import build_dataloaders

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
train_loader, val_loader = build_dataloaders(tokenizer, train_size=200000)

train batches: 998  val batches: 20


In [9]:
# # stage 2: sequence-level flow matching
# # z_data = decoder.compress(encoder_hidden) → [B, 128, 256] per-position latents
# # flow net learns noise [B*128, 256] → data [B*128, 256], position-wise

# flow_net  = FlowNet(latent_dim=256).to(device)
# optimizer = AdamW(flow_net.parameters(), lr=1e-4)

# EPOCHS    = 20
# best_loss = float("inf")

# for epoch in range(EPOCHS):
#     flow_net.train()
#     encoder.eval()

#     train_loss = 0

#     for step, batch in enumerate(train_loader):
#         input_ids      = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)

#         with torch.no_grad():
#             hidden = encoder(input_ids, attention_mask)  # [B, 128, 768]
#             z_data = decoder.compress(hidden)             # [B, 128, 256]

#         B, S, D = z_data.shape
#         z_flat  = z_data.view(B * S, D)                  # [B*128, 256]
#         z_noise = torch.randn_like(z_flat)

#         t   = torch.rand(B * S, device=device)
#         z_t = (1 - t.unsqueeze(-1)) * z_noise + t.unsqueeze(-1) * z_flat

#         v_true = z_flat - z_noise                        # [B*128, 256]
#         v_pred = flow_net(z_t, t)                        # [B*128, 256]

#         loss = F.mse_loss(v_pred, v_true)

#         optimizer.zero_grad()
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(flow_net.parameters(), max_norm=1.0)
#         optimizer.step()

#         train_loss += loss.item()

#         if step % 50 == 0:
#             print(f"epoch {epoch+1} step {step}/{len(train_loader)} | loss {loss.item():.4f}")

#     avg_loss = train_loss / len(train_loader)
#     print(f"\nepoch {epoch+1} done | avg loss {avg_loss:.4f}\n")

#     if avg_loss < best_loss:
#         best_loss = avg_loss
#         torch.save({"flow_net": flow_net.state_dict()}, "stage2_best.pt")
#         print(f"saved best model at loss {best_loss:.4f}")

In [10]:
import sys
import torch
import torch.nn as nn
sys.path.insert(0, ".")
from paralla_decoder import BertEncoder, ParallelDecoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FlowNet(nn.Module):
    def __init__(self, latent_dim=256, hidden_dim=2048, depth=8):
        super().__init__()
        layers = [nn.Linear(latent_dim + 1, hidden_dim), nn.SiLU()]
        for _ in range(depth - 2):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.SiLU()]
        layers.append(nn.Linear(hidden_dim, latent_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, z_t, t):
        inp = torch.cat([z_t, t.unsqueeze(-1)], dim=-1)
        return self.net(inp)

# init
encoder = BertEncoder().to(device)
decoder = ParallelDecoder(latent_dim=256).to(device)
flow_net = FlowNet(latent_dim=256, hidden_dim=2048, depth=8).to(device)

# load
ckpt = torch.load("stage2_best.pt", map_location=device, weights_only=False)
state_dict = ckpt["flow_net"]
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
flow_net.load_state_dict(state_dict)
encoder.load_state_dict(ckpt["encoder"])

flow_net.eval()
encoder.eval()
decoder.eval()

print("loaded successfully")
print(f"flow_net params: {sum(p.numel() for p in flow_net.parameters()):,}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18409.87it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7120.21it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  |

loaded successfully
flow_net params: 26,231,040


In [11]:
@torch.no_grad()
def generate(flow_net, decoder, tokenizer, steps=200, num_samples=5):
    flow_net.eval()
    decoder.eval()

    for i in range(num_samples):
        z_flat = torch.randn(128, 256, device=device)    # [128, 256]

        dt = 1.0 / steps
        for step in range(steps):
            t = torch.full((128,), step * dt, device=device)
            v = flow_net(z_flat, t)
            z_flat = z_flat + v * dt

        z_seq = z_flat.unsqueeze(0)                      # [1, 128, 256]

        x        = decoder.project_up(z_seq)             # [1, 128, 768]
        out      = decoder.bert(inputs_embeds=x)
        logits   = decoder.to_logits(out.last_hidden_state)
        pred_ids = logits.argmax(-1)

        text = tokenizer.decode(pred_ids[0].cpu(), skip_special_tokens=True)
        print(f"sample {i+1}: {text[:200]}")
        print()

generate(flow_net, decoder, tokenizer)

sample 1: ##wewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewe

sample 2: ##wewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewe

sample 3: ##wewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewe

sample 4: ##wewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewe

sample 5: ##wewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewewe

In [12]:
with torch.no_grad():
    batch = next(iter(train_loader))
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    hidden = encoder(input_ids, attention_mask)
    z_data = decoder.compress(hidden)
    print("z_data mean:", z_data.mean().item())
    print("z_data std: ", z_data.std().item())
    print("z_data min: ", z_data.min().item())
    print("z_data max: ", z_data.max().item())

z_data mean: 0.008496187627315521
z_data std:  0.27930957078933716
z_data min:  -1.6688326597213745
z_data max:  1.6538909673690796


In [13]:
with torch.no_grad():
    batch = next(iter(train_loader))
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    
    hidden = encoder(input_ids, attention_mask)   # [B, 128, 768]
    z_data = decoder.compress(hidden)              # [B, 128, 256]
    
    # manual decompress: z → 768 → bert → logits
    h_up = decoder.project_up(z_data)             # [B, 128, 768]
    bert_out = decoder.bert(inputs_embeds=h_up)
    logits = decoder.to_logits(bert_out.last_hidden_state)  # [B, 128, vocab]
    pred_ids = logits.argmax(-1)
    
    for i in range(2):
        original = tokenizer.decode(input_ids[i], skip_special_tokens=True)
        reconstructed = tokenizer.decode(pred_ids[i], skip_special_tokens=True)
        print(f"--- sample {i+1} ---")
        print(f"original:      {original[:100]}")
        print(f"reconstructed: {reconstructed[:100]}")
        print()

--- sample 1 ---
original:      the russian army has mounted some multiple rocket launchers on turretless t @ - @ 72 tanks and calle
reconstructed: ##wewewewewewewewewewewewewewewewewewewewewe tribunalwewewewewewewewewewewewewewewewewewewewewewewew

--- sample 2 ---
original:      on march 1, 2011, rihanna asked fans to help her choose the next single from loud using twitter, say
reconstructed: trimwewe trim google trim google trim trim trim trim trimwe trim trim trim trim trim trim trim trim 



In [14]:
# see all decoder methods
print([m for m in dir(decoder) if not m.startswith('_')])

['T_destination', 'add_module', 'apply', 'bert', 'bfloat16', 'buffers', 'call_super_init', 'children', 'compile', 'compress', 'cpu', 'cuda', 'decode_from_latent', 'double', 'dump_patches', 'eval', 'extra_repr', 'float', 'forward', 'get_buffer', 'get_extra_state', 'get_parameter', 'get_submodule', 'half', 'ipu', 'load_state_dict', 'modules', 'mtia', 'named_buffers', 'named_children', 'named_modules', 'named_parameters', 'parameters', 'project_up', 'register_backward_hook', 'register_buffer', 'register_forward_hook', 'register_forward_pre_hook', 'register_full_backward_hook', 'register_full_backward_pre_hook', 'register_load_state_dict_post_hook', 'register_load_state_dict_pre_hook', 'register_module', 'register_parameter', 'register_state_dict_post_hook', 'register_state_dict_pre_hook', 'requires_grad_', 'set_extra_state', 'set_submodule', 'share_memory', 'smart_apply', 'state_dict', 'to', 'to_empty', 'to_logits', 'train', 'training', 'type', 'xpu', 'zero_grad']


In [15]:
# see what's in the checkpoint
ckpt = torch.load("stage1_best.pt", map_location=device, weights_only=False)
print(ckpt.keys())
# if loss was saved:
if "loss" in ckpt:
    print("best loss:", ckpt["loss"])

dict_keys(['decoder', 'epoch', 'val_loss'])


In [16]:
import importlib
import paralla_decoder
importlib.reload(paralla_decoder)
from paralla_decoder import BertEncoder, ParallelDecoder

# reload models
encoder = BertEncoder().to(device)
decoder = ParallelDecoder(latent_dim=256).to(device)
ckpt = torch.load("stage1_best.pt", map_location=device, weights_only=False)
decoder.load_state_dict(ckpt["decoder"])
encoder.eval()
decoder.eval()

using: cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7791.81it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9656.47it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 

ParallelDecoder(
  (compress): Linear(in_features=768, out_features=256, bias=True)
  (project_up): Linear(in_features=256, out_features=768, bias=True)
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
         

In [18]:
with torch.no_grad():
    batch = next(iter(val_loader))
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    
    hidden = encoder(input_ids, attention_mask)
    z_data = decoder.compress(hidden)
    
    logits = decoder.decode_from_latent(z_data)
    pred_ids = logits.argmax(-1)
    
    for i in range(3):
        original = tokenizer.decode(input_ids[i], skip_special_tokens=True)
        reconstructed = tokenizer.decode(pred_ids[i], skip_special_tokens=True)
        print(f"--- sample {i+1} ---")
        print(f"original:      {original[:120]}")
        print(f"reconstructed: {reconstructed[:120]}")
        print()

--- sample 1 ---
original:      = homarus gammarus =
reconstructed: = homarus gammarus = “ “ “ “ “ “ “ “ “ “ — “ = “ — “ — “ “ — “ — — — “ — — “ … “ “ “ “ “ “ “ “ “ “ = “ “ “ — — — — “ “ “

--- sample 2 ---
original:      homarus gammarus, known as the european lobster or common lobster, is a species of clawed lobster from the eastern atlan
reconstructed: homarus gammarus, known as the european lobster or common lobster, is a species of clawed lobster from the eastern atlan

--- sample 3 ---
original:      = = description = =
reconstructed: = = description = = accolades constitution accolades terminology overview — accolades lawsuit “ = “ – = = = “ “ = – – “ 

